# Mens vs Womens Email by Segment

Supplements stage 02 to answer challenge questions 5 and 8 directly:

5. Did the Mens email perform differently from the Womens email across customer segments?
8. Which audience should receive each email, and what data supports that?

**This analysis is exploratory.** It is not in the analysis plan, and results by segment from stages 02 to 04, including both emails' effects in the stage 03 cells, had been seen before it was designed. Holm correction limits false positives within each question, but it does not make the analysis confirmatory.

## Method

- **Data:** all 64,000 customers. No train-test split, because no rule is learned: the decision criteria below are fixed.
- **Question 5:** the six segment splits of stage 02, unchanged. Each regression is `spend ~ mens_email * segment` among emailed customers, with HC3 standard errors. **The interaction is the change in the Mens email's lead over the Womens email.** A split counts as different only if its Holm-adjusted p-value (six tests) is below 0.05.
- **Question 8:** three audiences that do not overlap: bought men's only, women's only, or both. **The default is the Mens email, the winner on average in stage 01. An audience switches to the Womens email only if the Womens lead is significant after Holm correction (three tests).**
- **Worth sending:** each email in each audience is also classified with the rule of 2026-11-06: worth sending if the 95% interval's lower end is above \$0.02, uncertain if only the estimate is, and not worth sending otherwise. **If the Womens email must be used, it is recommended only for audiences where it is worth sending.**
- **Secondary metrics:** visit and conversion are descriptive and decide nothing.
- **Wording:** a difference that is not significant is reported as "not shown", never as "the same".

In [1]:
import sys

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

sys.path.append("..")
from src.prepare import load_data

ALPHA, BREAK_EVEN = 0.05, 0.008 / 0.40

df = load_data()
df["recent"] = (df["recency"] <= 3).astype(int)
df["online"] = (df["channel"] != "Phone").astype(int)
df["steps"] = df["history_segment"].str[0].astype(int).clip(upper=5) - 1
df["urban"] = df["zip_code"].map({"Rural": 0, "Urban": 1})  # Suburban is left out of this test, as in H5
emailed = df[df["segment"] != "No E-Mail"]

SPLITS = {
    "Recency: 1-3 vs 4-12 months": "recent",
    "Bought women's: yes vs no": "womens",
    "Bought men's: yes vs no": "mens",
    "Channel: Web or Multichannel vs Phone": "online",
    "Past-year spend: per level": "steps",
    "Zip code: Urban vs Rural": "urban",
}
AUDIENCES = ["men's only", "women's only", "both"]
pd.crosstab(df["bought"], df["segment"]).loc[AUDIENCES]

segment,Mens E-Mail,No E-Mail,Womens E-Mail
bought,,,
men's only,9558,9638,9622
women's only,9568,9519,9647
both,2181,2149,2118


## Question 5: does the Mens email's lead change across segments?

In [2]:
def lead_interaction(metric, column):
    """Change in the Mens email's lead over the Womens email across one segment split."""
    fit = smf.ols(f"{metric} ~ mens_email * {column}", emailed.dropna(subset=[column])).fit(cov_type="HC3")
    term = f"mens_email:{column}"
    low, high = fit.conf_int().loc[term]
    return {"change_in_lead": fit.params[term], "low": low, "high": high,
            "standard_error": fit.bse[term], "p_value": fit.pvalues[term]}


q5 = pd.DataFrame({name: lead_interaction("spend", column) for name, column in SPLITS.items()}).T
q5["holm_p"] = multipletests(q5["p_value"], method="holm")[1]
q5["min_detectable"] = (norm.ppf(1 - ALPHA / (2 * len(q5))) + norm.ppf(0.8)) * q5["standard_error"]
q5["verdict"] = np.where(q5["holm_p"] < ALPHA, "different", "difference not shown")
q5.round(3)

,change_in_lead,low,high,standard_error,p_value,holm_p,min_detectable,verdict
Recency: 1-3 vs 4-12 months,0.529,-0.179,1.237,0.361,0.143,0.714,1.257,difference not shown
Bought women's: yes vs no,-0.107,-0.739,0.524,0.322,0.739,1.000,1.121,difference not shown
Bought men's: yes vs no,0.656,0.041,1.271,0.314,0.037,0.220,1.092,difference not shown
Channel: Web or Multichannel vs Phone,0.019,-0.596,0.634,0.314,0.952,1.000,1.092,difference not shown
Past-year spend: per level,0.187,-0.072,0.446,0.132,0.157,0.714,0.459,difference not shown
Zip code: Urban vs Rural,-0.173,-1.115,0.769,0.481,0.718,1.000,1.673,difference not shown


**Reading the result:** **none of the six splits shows a difference in the Mens email's lead on spend.** The smallest p-value, 0.037 for bought men's merchandise, becomes 0.22 after Holm correction. Its printed interval excludes zero only because that interval is not corrected for six tests. The test could reliably detect only changes of about \$1.09 to \$1.67 (about \$0.46 per spend level), against an average lead of \$0.35, so "not shown" does not mean "the same".

### Both emails within each segment

Descriptive. A lead that is significant in one segment and not in another does not show that the segments differ; only the table above tests that.

In [3]:
def classify(estimate, low):
    if low > BREAK_EVEN:
        return "worth sending"
    if estimate > BREAK_EVEN:
        return "uncertain"
    return "not worth sending"


def email_effects(data):
    """Each email against no email, and the Mens email's lead, within one group of customers."""
    fit = smf.ols("spend ~ mens_email + womens_email", data).fit(cov_type="HC3")
    ci = fit.conf_int()
    row = {"customers": len(data)}
    for name, term in [("mens", "mens_email"), ("womens", "womens_email")]:
        row[name] = fit.params[term]
        row[f"{name}_low"], row[f"{name}_high"] = ci.loc[term]
        row[f"{name}_category"] = classify(row[name], row[f"{name}_low"])
    test = fit.t_test("mens_email - womens_email = 0")
    row["lead"] = float(test.effect[0])
    row["lead_low"], row["lead_high"] = test.conf_int()[0]
    row["lead_p"] = float(test.pvalue)
    return row


by_segment = pd.DataFrame({
    (name, int(value)): email_effects(data)
    for name, column in SPLITS.items()
    for value, data in df.dropna(subset=[column]).groupby(column)
}).T
by_segment[["customers", "mens", "womens", "lead", "lead_low", "lead_high"]].astype(float).round(2)

customers  mens  womens  lead  \
Recency: 1-3 vs 4-12 months           0    41607.0  0.62    0.46  0.16   
                                      1    22393.0  1.05    0.36  0.69   
Bought women's: yes vs no             0    28818.0  0.68    0.28  0.40   
                                      1    35182.0  0.84    0.54  0.30   
Bought men's: yes vs no               0    28734.0  0.62    0.63 -0.02   
                                      1    35266.0  0.90    0.26  0.64   
Channel: Web or Multichannel vs Phone 0    28021.0  0.56    0.23  0.33   
                                      1    35979.0  0.93    0.58  0.35   
Past-year spend: per level            0    22970.0  0.54    0.63 -0.09   
                                      1    14254.0  0.70    0.46  0.25   
                                      2    12289.0  0.54   -0.32  0.86   
                                      3     6409.0  1.57   -0.10  1.67   
                                      4     8078.0  1.27    1.35 -0.08   
Zip code: Urban vs Rural              0     9563.0  0.76    0.51  0.24   
                                      1    25661.0  0.74    0.67  0.07   

                                         lead_low  lead_high  
Recency: 1-3 vs 4-12 months           0     -0.19       0.51  
                                      1      0.07       1.31  
Bought women's: yes vs no             0     -0.07       0.88  
                                      1     -0.12       0.71  
Bought men's: yes vs no               0     -0.42       0.39  
                                      1      0.18       1.10  
Channel: Web or Multichannel vs Phone 0     -0.09       0.75  
                                      1     -0.10       0.80  
Past-year spend: per level            0     -0.59       0.41  
                                      1     -0.34       0.83  
                                      2      0.24       1.48  
                                      3      0.48       2.86  
                                      4     -1.22       1.06  
Zip code: Urban vs Rural              0     -0.56       1.04  
                                      1     -0.43       0.57

**Reading the result:** **these rows show where each change in lead comes from, but they are not tests of differences between segments.** For example, the lead is \$0.64 among men's buyers and -\$0.02 among the others, and the change between them, \$0.66, is what the first table tests. Several single-segment intervals exclude zero, including spend steps 2 and 3 (\$200-\$500 of past-year spend), but none of the corresponding splits passes that test.

## Question 8: which audience should receive each email?

In [4]:
q8 = pd.DataFrame({name: email_effects(df[df["bought"] == name]) for name in AUDIENCES}).T
q8["lead_holm_p"] = multipletests(q8["lead_p"].astype(float), method="holm")[1]


def recommend(row):
    if row["lead_holm_p"] < ALPHA:
        return "Mens (shown better)" if row["lead"] > 0 else "Womens (shown better)"
    return "Mens (default, neither shown better)"


q8["email_to_send"] = q8.apply(recommend, axis=1)
q8["womens_if_it_must_be_used"] = np.where(q8["womens_category"] == "worth sending", "yes", "not recommended")

display(q8[["customers", "mens", "mens_low", "mens_high", "mens_category",
            "womens", "womens_low", "womens_high", "womens_category"]])
q8[["lead", "lead_low", "lead_high", "lead_p", "lead_holm_p", "email_to_send", "womens_if_it_must_be_used"]]

,customers,mens,mens_low,mens_high,mens_category,womens,womens_low,womens_high,womens_category
men's only,28818,0.682735,0.247149,1.118321,worth sending,0.278354,-0.123583,0.680292,uncertain
women's only,28734,0.615731,0.26966,0.961802,worth sending,0.631648,0.295116,0.96818,worth sending
both,6448,1.824986,0.489854,3.160119,worth sending,0.161881,-0.813566,1.137328,uncertain


,lead,lead_low,lead_high,lead_p,lead_holm_p,email_to_send,womens_if_it_must_be_used
men's only,0.404381,-0.071927,0.880689,0.096114,0.192229,"Mens (default, neither shown better)",not recommended
women's only,-0.015917,-0.424532,0.392697,0.939142,0.939142,"Mens (default, neither shown better)",yes
both,1.663105,0.331622,2.994589,0.014361,0.043082,Mens (shown better),not recommended


**Reading the result:** **the Mens email is the one to send in all three audiences, and it is worth sending in each.**

- **Men's only:** the lead of \$0.40 is not significant, so the Mens email is the default. The Womens email is uncertain.
- **Women's only:** neither email is shown better (lead -\$0.02, 95% CI -\$0.42 to \$0.39), so the Mens email is the default. **The Womens email is also worth sending, and this is the only audience where it is.**
- **Both:** the Mens email is better (lead \$1.66, Holm p = 0.043), narrowly. The Womens email is uncertain.

The \$0.02 break-even applies to an email against no email. A difference between two emails is compared with zero, because their costs cancel out. The result for buyers of both lines says which email suits them, not that they respond more than other audiences.

## Secondary metrics

Change in the Mens email's lead, in percentage points, with uncorrected intervals. These are descriptive and decide neither question.

In [5]:
secondary = pd.DataFrame({
    (metric, name): {key: value * 100 for key, value in lead_interaction(metric, column).items()
                     if key in ("change_in_lead", "low", "high")}
    for metric in ["visit", "conversion"]
    for name, column in SPLITS.items()
}).T
secondary.round(2)

change_in_lead   low  high
visit      Recency: 1-3 vs 4-12 months                      1.49 -0.05  3.02
           Bought women's: yes vs no                       -4.86 -6.26 -3.46
           Bought men's: yes vs no                          6.32  4.90  7.74
           Channel: Web or Multichannel vs Phone            0.02 -1.39  1.42
           Past-year spend: per level                       0.48 -0.05  1.01
           Zip code: Urban vs Rural                        -1.16 -3.42  1.10
conversion Recency: 1-3 vs 4-12 months                      0.14 -0.30  0.57
           Bought women's: yes vs no                       -0.31 -0.69  0.08
           Bought men's: yes vs no                          0.56  0.17  0.95
           Channel: Web or Multichannel vs Phone           -0.01 -0.40  0.37
           Past-year spend: per level                       0.13 -0.03  0.29
           Zip code: Urban vs Rural                         0.11 -0.52  0.74

**Reading the result:** **the Mens email's lead in visits changes with past purchases.** It is 6.3 points larger among men's buyers and 4.9 points smaller among women's buyers, and both intervals exclude zero. For conversion, only the interval for men's buyers excludes zero (+0.56 points). These metrics are uncorrected and descriptive, and do not change either answer.

## Checks added after the results were seen

These checks were not among the criteria fixed before the run. They show how fragile the results are and change no verdict.

### Intervals corrected for the number of comparisons

Bonferroni intervals for the six question 5 tests, and for the six effects classified as worth sending or not in question 8.

In [6]:
z_six = norm.ppf(1 - ALPHA / (2 * 6))
q5_corrected = pd.DataFrame({
    "change_in_lead": q5["change_in_lead"],
    "low": (q5["change_in_lead"] - z_six * q5["standard_error"]).astype(float),
    "high": (q5["change_in_lead"] + z_six * q5["standard_error"]).astype(float),
})
display(q5_corrected.astype(float).round(2))

rows = {}
for name in AUDIENCES:
    fit = smf.ols("spend ~ mens_email + womens_email", df[df["bought"] == name]).fit(cov_type="HC3")
    for email, term in [("Mens", "mens_email"), ("Womens", "womens_email")]:
        low = fit.conf_int(ALPHA / 6).loc[term, 0]
        rows[(name, email)] = {"corrected_low": round(low, 3), "category": classify(fit.params[term], low)}
pd.DataFrame(rows).T

,change_in_lead,low,high
Recency: 1-3 vs 4-12 months,0.53,-0.42,1.48
Bought women's: yes vs no,-0.11,-0.96,0.74
Bought men's: yes vs no,0.66,-0.17,1.48
Channel: Web or Multichannel vs Phone,0.02,-0.81,0.85
Past-year spend: per level,0.19,-0.16,0.54
Zip code: Urban vs Rural,-0.17,-1.44,1.09


corrected_low       category
men's only   Mens           0.096  worth sending
             Womens        -0.263      uncertain
women's only Mens            0.15  worth sending
             Womens         0.179  worth sending
both         Mens           0.028  worth sending
             Womens        -1.151      uncertain

**Reading the result:** **all six question 5 intervals include zero once corrected, matching the Holm verdicts, and all four worth-sending categories hold.** The Mens email among buyers of both lines holds only narrowly (lower end \$0.028 against \$0.02).

### How fragile is the one significant lead?

In [7]:
from scipy import stats

both = df[df["bought"] == "both"]
mens_spend = np.sort(both.loc[both["segment"] == "Mens E-Mail", "spend"].to_numpy())
womens_spend = both.loc[both["segment"] == "Womens E-Mail", "spend"].to_numpy()
print(f"Buyers: {(mens_spend > 0).sum()} with the Mens email, {(womens_spend > 0).sum()} with the Womens email")

rng = np.random.default_rng(42)
pooled, n_mens = np.concatenate([mens_spend, womens_spend]), len(mens_spend)
observed = abs(mens_spend.mean() - womens_spend.mean())
hits = 0
for _ in range(20_000):
    shuffled = rng.permutation(pooled)
    hits += abs(shuffled[:n_mens].mean() - shuffled[n_mens:].mean()) >= observed - 1e-12
other_p = q8.loc[["men's only", "women's only"], "lead_p"].astype(float).tolist()
print(f"Permutation test: Holm p = {multipletests([hits / 20_000] + other_p, method='holm')[1][0]:.3f}")

for k in [1, 2, 3]:
    p = stats.ttest_ind(mens_spend[:-k], womens_spend, equal_var=False).pvalue
    print(f"Without the {k} largest Mens purchase(s): lead {mens_spend[:-k].mean() - womens_spend.mean():.2f}, "
          f"Holm p = {multipletests([p] + other_p, method='holm')[1][0]:.3f}")

nine = q5["p_value"].astype(float).tolist() + q8["lead_p"].astype(float).tolist()
print(f"All nine tests corrected as one family: Holm p = {multipletests(nine, method='holm')[1][-1]:.3f}")

Buyers: 53 with the Mens email, 31 with the Womens email


Permutation test: Holm p = 0.041
Without the 1 largest Mens purchase(s): lead 1.44, Holm p = 0.075
Without the 2 largest Mens purchase(s): lead 1.22, Holm p = 0.130
Without the 3 largest Mens purchase(s): lead 1.02, Holm p = 0.218
All nine tests corrected as one family: Holm p = 0.129


**Reading the result:** **the lead among buyers of both lines is fragile.** A permutation test agrees with it, so it does not come from the normal approximation. But it rests on 53 and 31 buyers: without the single largest Mens purchase, Holm p rises to about 0.075, and with all nine tests corrected as one family it is 0.13. It is reported as a fragile exploratory result.

**Summary: the Mens email is the default for every audience. If the Womens email must be used, send it to customers who bought only women's merchandise.**